# HALO Option 1 — Multilingual vs. Multi-domain NMT

**Executed artifact:** deterministic zero-shot translation for
English→Gujarati, Georgian, Tamil, and Simplified Chinese.

**Proposed artifact:** an equal-budget LoRA comparison of multilingual
and multi-domain adaptation. The LoRA comparison is designed and
manifested here but is **not trained** in this demonstration.

The notebook uses NLLB as a compact, common experimental instrument—not
as a claim about the 2026 state of the art. Translation supplies a
controlled source-grounding context; these measurements do not establish
that MT hallucinations generalize to open-world LLM hallucinations.

Run all cells in order. In Colab, select a GPU runtime first.

## 1. Install and report pinned runtime versions

In [1]:
import importlib.metadata as metadata
import subprocess
import sys

PINS = {
    "transformers": "4.53.0",
    "huggingface-hub": "0.33.1",
    "sentencepiece": "0.2.1",
    "sacrebleu": "2.5.1",
    "pandas": "2.3.1",
    "requests": "2.34.2",
    "accelerate": "1.9.0",
    "peft": "0.16.0",
    "nbformat": "5.10.4",
    "nbclient": "0.10.2",
}

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

mismatches = [
    f"{package}=={version}"
    for package, version in PINS.items()
    if installed_version(package) != version
]
if mismatches:
    print("Installing pinned packages:", ", ".join(mismatches))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *mismatches]
    )

import platform
import torch

RUNTIME_VERSIONS = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    **{name: installed_version(name) for name in PINS},
}
print(RUNTIME_VERSIONS)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Installing pinned packages: sentencepiece==0.2.1, sacrebleu==2.5.1, pandas==2.3.1, accelerate==1.9.0, peft==0.16.0


{'python': '3.13.5', 'platform': 'macOS-26.5.2-arm64-arm-64bit-Mach-O', 'torch': '2.10.0', 'transformers': '4.53.0', 'huggingface-hub': '0.33.1', 'sentencepiece': '0.2.1', 'sacrebleu': '2.5.1', 'pandas': '2.3.1', 'requests': '2.34.2', 'accelerate': '1.9.0', 'peft': '0.16.0', 'nbformat': '5.10.4', 'nbclient': '0.10.2'}
CUDA available: False


## 2. Configuration, immutable revisions, and data contracts

In [2]:
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import csv
import hashlib
import json
import math
import os
import random
import re
import shutil
import tempfile
import time
import unicodedata
import zipfile

# The standard HTTP path is more reliable than the optional Xet client
# in short-lived Colab/local kernels and preserves the same pinned files.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
# This notebook is PyTorch-only. Prevent an unrelated local TensorFlow
# installation from being imported through optional Transformers paths.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")

import pandas as pd
import requests
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF
from huggingface_hub import snapshot_download
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed

SEED = 42
TRAINING_SEEDS = [13, 42, 73]
MAX_SOURCE_TOKENS = 256
MAX_TARGET_TOKENS = 256
NUM_BEAMS = 4

MODEL_ID = "facebook/nllb-200-distilled-600M"
MODEL_REVISION = "f8d333a098d19b4fd9a8b18f94170487ad3f821d"
MODEL_WEIGHTS_SHA256 = (
    "c266c2cfd19758b6d09c1fc31ecdf1e485509035f6b51dfe84f1ada83eefcc42"
)
MODEL_FILES_SHA256 = {
    "config.json": "f9b4081d3d108e9d06532ea8d92b932c04ed3a15ff2f87e6982f8e14db51fbc5",
    "generation_config.json": "0bb604cdb8392649176935a447ce994128f170a6908c135a17d0e3dfb8113cb2",
    "pytorch_model.bin": MODEL_WEIGHTS_SHA256,
    "sentencepiece.bpe.model": "14bb8dfb35c0ffdea7bc01e56cea38b9e3d5efcdcb9c251d6b40538e1aab555a",
    "special_tokens_map.json": "992bd4ed610d644d6823081937bcc91bb8878dd556cea4ae5327f2480361330e",
    "tokenizer.json": "e316b82de11d0f951f370943b3c438311629547285129b0b81dadabd01bca665",
    "tokenizer_config.json": "d1aa8c3697d3e35674f97b5b7e9c99d22b010f528e80140257d97316be90d044",
}
SOURCE_LANGUAGE_CODE = "eng_Latn"

TARGETS = {
    "guj_Gujr": {"name": "Gujarati", "ntrex_suffix": "guj", "bleu_tokenizer": "intl"},
    "kat_Geor": {"name": "Georgian", "ntrex_suffix": "kat", "bleu_tokenizer": "intl"},
    "tam_Taml": {"name": "Tamil", "ntrex_suffix": "tam", "bleu_tokenizer": "intl"},
    "zho_Hans": {
        "name": "Simplified Chinese",
        "ntrex_suffix": "zho-CN",
        "bleu_tokenizer": "zh",
    },
}

NTREX_REVISION = "468c6b69c7f6a75d31d4743d9daba2af566cc18d"
WMT24PP_REVISION = "e65f5856b1de3319c748c15e5aec0bc2336ec3b0"

NTREX_BASE = (
    "https://raw.githubusercontent.com/MicrosoftTranslator/NTREX/"
    f"{NTREX_REVISION}/"
)
WMT24PP_URL = (
    "https://huggingface.co/datasets/google/wmt24pp/resolve/"
    f"{WMT24PP_REVISION}/en-zh_CN.jsonl?download=true"
)
TICO_URL = "https://tico-19.github.io/data/tico19-testset.zip"

DATA_FILES = {
    "ntrex_documents": {
        "url": NTREX_BASE + "DOCUMENT_IDS.tsv",
        "sha256": "88a8b0ff1eae47e83f60dcde01075b8c87d75ced59ca09e0af1eb539958e05d5",
    },
    "ntrex_source": {
        "url": NTREX_BASE + "NTREX-128/newstest2019-src.eng.txt",
        "sha256": "389e8f5796c66db4f646dfad33e1ec622d74767af5ef112b42a1f2cd814df3cc",
    },
    "ntrex_guj": {
        "url": NTREX_BASE + "NTREX-128/newstest2019-ref.guj.txt",
        "sha256": "fbe9554a4e71ca9a4367bdd1bba3b5bd3067653ca39ce8492dca3d8f865878e5",
    },
    "ntrex_kat": {
        "url": NTREX_BASE + "NTREX-128/newstest2019-ref.kat.txt",
        "sha256": "711bd756cdd62d060c02d1adf162951f952d9061722b249ecf7300df9af11b2f",
    },
    "ntrex_tam": {
        "url": NTREX_BASE + "NTREX-128/newstest2019-ref.tam.txt",
        "sha256": "dbc64d8191738c0bad1b6ee0e4357edfa18a02dedd15cb523ae4c9205dce55eb",
    },
    "ntrex_zho-CN": {
        "url": NTREX_BASE + "NTREX-128/newstest2019-ref.zho-CN.txt",
        "sha256": "8bcd6f0868d33f26c8da12f55a82f66088a7b60a5f7b0b1a99dcec51daa8054a",
    },
    "wmt24pp_zh": {
        "url": WMT24PP_URL,
        "sha256": "984ec4da714800aae2b9ee6e2601d1cadb01a770e01ed385d5283ce7a0585287",
    },
    "tico19_archive": {
        "url": TICO_URL,
        "sha256": "0e82fc7ceaa877606c8934f32ead185716c9d2c54b3818f2a7ae655f5e7a08d8",
    },
}

def locate_project_root():
    cwd = Path.cwd().resolve()
    if (cwd / "notebooks").is_dir() and (cwd / "report").is_dir():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "report").is_dir():
        return cwd.parent
    return cwd

PROJECT_ROOT = locate_project_root()
IN_COLAB = "google.colab" in sys.modules
RESULTS_DIR = (
    Path("/content/halo_option1_results")
    if IN_COLAB
    else PROJECT_ROOT / "results"
)
DATA_DIR = Path(tempfile.gettempdir()) / "halo_option1_demo_data"
MODEL_CACHE_DIR = Path(tempfile.gettempdir()) / "halo_option1_model_cache"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

local_model_override = os.environ.get("HALO_LOCAL_MODEL_DIR")
# A local override is accepted only after every model/tokenizer file has
# matched the byte hashes of the pinned Hugging Face revision.

REDUCED_DEMO = os.environ.get("HALO_REDUCED_DEMO", "0") == "1"
SAMPLE_COUNTS = {
    "ntrex_per_language": 5 if REDUCED_DEMO else 25,
    "wmt_per_domain": 3 if REDUCED_DEMO else 15,
    "tico": 5 if REDUCED_DEMO else 25,
    "counterfactual_pairs": 3 if REDUCED_DEMO else 10,
}
print("Project root: repository root")
print("Results:", RESULTS_DIR.relative_to(PROJECT_ROOT))
print("Reduced fallback:", REDUCED_DEMO)
print("Sample counts:", SAMPLE_COUNTS)

Project root: repository root
Results: results
Reduced fallback: False
Sample counts: {'ntrex_per_language': 25, 'wmt_per_domain': 15, 'tico': 25, 'counterfactual_pairs': 10}


## 3. Download, filter, normalize, and validate the datasets

In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def download_checked(url: str, destination: Path, expected_sha256: str) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and sha256_file(destination) == expected_sha256:
        return destination
    if destination.exists():
        destination.unlink()

    temporary = destination.with_suffix(destination.suffix + ".part")
    if temporary.exists():
        temporary.unlink()
    headers = {"User-Agent": "halo-option1-research-demo/1.0"}
    with requests.get(url, stream=True, timeout=(20, 180), headers=headers) as response:
        response.raise_for_status()
        with temporary.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
    actual = sha256_file(temporary)
    if actual != expected_sha256:
        temporary.unlink()
        raise AssertionError(
            f"Checksum mismatch for {url}: expected {expected_sha256}, got {actual}"
        )
    temporary.replace(destination)
    return destination


LOCAL_DATA = {}
for key, specification in DATA_FILES.items():
    suffix = ".zip" if key == "tico19_archive" else (".jsonl" if "wmt" in key else ".txt")
    LOCAL_DATA[key] = download_checked(
        specification["url"],
        DATA_DIR / f"{key}{suffix}",
        specification["sha256"],
    )

tico_member = "tico19-testset/test/test.en-zh.tsv"
tico_path = DATA_DIR / "tico19-test.en-zh.tsv"
with zipfile.ZipFile(LOCAL_DATA["tico19_archive"]) as archive:
    assert tico_member in archive.namelist()
    if not tico_path.exists():
        with archive.open(tico_member) as source, tico_path.open("wb") as target:
            shutil.copyfileobj(source, target)

print("Verified downloads:")
for key, path in LOCAL_DATA.items():
    print(f"  {key}: {path.name} ({path.stat().st_size:,} bytes)")

Verified downloads:
  ntrex_documents: ntrex_documents.txt (30,754 bytes)
  ntrex_source: ntrex_source.txt (251,739 bytes)
  ntrex_guj: ntrex_guj.txt (647,954 bytes)
  ntrex_kat: ntrex_kat.txt (718,029 bytes)
  ntrex_tam: ntrex_tam.txt (851,931 bytes)
  ntrex_zho-CN: ntrex_zho-CN.txt (241,603 bytes)
  wmt24pp_zh: wmt24pp_zh.jsonl (1,061,605 bytes)
  tico19_archive: tico19_archive.zip (15,168,404 bytes)


In [4]:
WHITESPACE_RE = re.compile(r"\s+")

def normalize_text(text: str) -> str:
    return WHITESPACE_RE.sub(
        " ", unicodedata.normalize("NFKC", str(text)).strip()
    )


def normalized_source_hash(text: str) -> str:
    normalized = normalize_text(text).casefold()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def rank_key(namespace: str, value: str) -> str:
    payload = f"{SEED}|{namespace}|{value}".encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def document_split_map(document_ids, namespace: str):
    documents = sorted(
        set(document_ids), key=lambda value: rank_key(namespace, value)
    )
    assert len(documents) >= 3, f"Too few documents for {namespace}"
    n_train = math.floor(0.70 * len(documents))
    n_validation = max(1, math.floor(0.10 * len(documents)))
    n_test = len(documents) - n_train - n_validation
    assert n_train > 0 and n_validation > 0 and n_test > 0
    return {
        **{doc: "train" for doc in documents[:n_train]},
        **{
            doc: "validation"
            for doc in documents[n_train : n_train + n_validation]
        },
        **{doc: "test" for doc in documents[n_train + n_validation :]},
    }


def deduplicate_by_source(rows):
    retained = {}
    for row in rows:
        source_hash = normalized_source_hash(row["source"])
        retained.setdefault(source_hash, row)
    return list(retained.values())


# NTREX: retain the earlier of its one normalized duplicate source and
# apply one shared document split across all four target languages.
ntrex_documents = LOCAL_DATA["ntrex_documents"].read_text("utf-8").splitlines()
ntrex_sources = LOCAL_DATA["ntrex_source"].read_text("utf-8").splitlines()
ntrex_references = {
    code: LOCAL_DATA[f"ntrex_{metadata['ntrex_suffix']}"]
    .read_text("utf-8")
    .splitlines()
    for code, metadata in TARGETS.items()
}
ntrex_lengths = {
    len(ntrex_documents),
    len(ntrex_sources),
    *(len(values) for values in ntrex_references.values()),
}
assert ntrex_lengths == {1997}, ntrex_lengths
ntrex_split = document_split_map(ntrex_documents, "ntrex|news")

first_index_by_source_hash = {}
retained_ntrex_indices = []
for index, source in enumerate(ntrex_sources):
    source_hash = normalized_source_hash(source)
    if source_hash not in first_index_by_source_hash:
        first_index_by_source_hash[source_hash] = index
        retained_ntrex_indices.append(index)
assert len(retained_ntrex_indices) == 1996

ntrex_rows = []
for code, references in ntrex_references.items():
    for index in retained_ntrex_indices:
        source = normalize_text(ntrex_sources[index])
        reference = normalize_text(references[index])
        assert source and reference
        ntrex_rows.append(
            {
                "example_id": f"ntrex:{index:04d}:{code}",
                "segment_index": index,
                "document_id": ntrex_documents[index],
                "source": source,
                "reference": reference,
                "target_language": code,
                "domain": "news",
                "dataset": "ntrex128",
                "split": ntrex_split[ntrex_documents[index]],
            }
        )

# WMT24++: independently assert both exclusions, then retain the lower
# segment_id for its one normalized duplicate source.
raw_wmt_rows = [
    json.loads(line)
    for line in LOCAL_DATA["wmt24pp_zh"].read_text("utf-8").splitlines()
    if line.strip()
]
assert len(raw_wmt_rows) == 998
required_wmt_fields = {
    "lp",
    "domain",
    "document_id",
    "segment_id",
    "is_bad_source",
    "source",
    "target",
    "original_target",
}
assert all(required_wmt_fields <= set(row) for row in raw_wmt_rows)
pre_dedup_wmt = [
    row
    for row in raw_wmt_rows
    if row["domain"] != "canary" and row["is_bad_source"] is False
]
assert len(pre_dedup_wmt) == 960
assert all(row["domain"] != "canary" for row in pre_dedup_wmt)
assert all(row["is_bad_source"] is False for row in pre_dedup_wmt)

pre_dedup_wmt.sort(key=lambda row: row["segment_id"])
retained_wmt = {}
for row in pre_dedup_wmt:
    retained_wmt.setdefault(normalized_source_hash(row["source"]), row)
dedup_wmt = list(retained_wmt.values())
assert len(dedup_wmt) == 959

wmt_split_maps = {
    domain: document_split_map(
        [row["document_id"] for row in dedup_wmt if row["domain"] == domain],
        f"wmt24pp|{domain}",
    )
    for domain in ("news", "social", "speech", "literary")
}
wmt_rows = []
for row in dedup_wmt:
    source = normalize_text(row["source"])
    reference = normalize_text(row["target"])
    assert source and reference
    wmt_rows.append(
        {
            "example_id": f"wmt24pp:{row['segment_id']:04d}:zho_Hans",
            "document_id": row["document_id"],
            "source": source,
            "reference": reference,
            "target_language": "zho_Hans",
            "domain": row["domain"],
            "dataset": "wmt24pp",
            "split": wmt_split_maps[row["domain"]][row["document_id"]],
        }
    )

# TICO-19: preserve its official test split. The document prefix is a
# transparent derived grouping key, not an official document field.
tico_rows = []
with tico_path.open(encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle, delimiter="\t")
    expected_fields = [
        "sourceLang",
        "targetLang",
        "sourceString",
        "targetString",
        "stringID",
        "url",
        "license",
        "translator_ID",
    ]
    assert reader.fieldnames == expected_fields
    for row in reader:
        assert row["sourceLang"].strip() == "en"
        assert row["targetLang"].strip() == "zh"
        string_id = row["stringID"].strip()
        source = normalize_text(row["sourceString"])
        reference = normalize_text(row["targetString"])
        assert string_id and source and reference
        tico_rows.append(
            {
                "example_id": f"tico19:{string_id}",
                "document_id": string_id.rsplit(":", 1)[0],
                "source": source,
                "reference": reference,
                "target_language": "zho_Hans",
                "domain": "medical",
                "dataset": "tico19",
                "split": "official_test",
            }
        )
assert len(tico_rows) == 2100
assert len({row["example_id"] for row in tico_rows}) == 2100

# Cross-split leakage is checked within each corpus. Repeated aligned
# NTREX sources across target languages must remain in the same split.
for dataset_name, rows in (
    ("ntrex128", ntrex_rows),
    ("wmt24pp", wmt_rows),
):
    splits_by_hash = defaultdict(set)
    documents_by_split = defaultdict(set)
    for row in rows:
        splits_by_hash[normalized_source_hash(row["source"])].add(row["split"])
        documents_by_split[row["split"]].add(row["document_id"])
    assert all(len(splits) == 1 for splits in splits_by_hash.values())
    split_names = list(documents_by_split)
    for left_index, left in enumerate(split_names):
        for right in split_names[left_index + 1 :]:
            assert documents_by_split[left].isdisjoint(documents_by_split[right])

print("Normalized rows:", {
    "NTREX": len(ntrex_rows),
    "WMT24++": len(wmt_rows),
    "TICO-19": len(tico_rows),
})
print("NTREX document split:", Counter(ntrex_split.values()))
print("WMT document splits:", {
    domain: dict(Counter(mapping.values()))
    for domain, mapping in wmt_split_maps.items()
})

Normalized rows: {'NTREX': 7984, 'WMT24++': 959, 'TICO-19': 2100}
NTREX document split: Counter({'train': 86, 'test': 25, 'validation': 12})
WMT document splits: {'news': {'train': 11, 'validation': 1, 'test': 5}, 'social': {'train': 23, 'validation': 3, 'test': 8}, 'speech': {'train': 77, 'validation': 11, 'test': 23}, 'literary': {'train': 5, 'validation': 1, 'test': 2}}


## 4. Token limits, proposed arm manifests, and deterministic demo sample

In [5]:
if local_model_override:
    MODEL_SNAPSHOT_DIR = Path(local_model_override).resolve()
    MODEL_SNAPSHOT_SOURCE = "verified_local_override"
else:
    MODEL_SNAPSHOT_DIR = Path(
        snapshot_download(
            repo_id=MODEL_ID,
            revision=MODEL_REVISION,
            cache_dir=MODEL_CACHE_DIR,
            allow_patterns=sorted(MODEL_FILES_SHA256),
        )
    ).resolve()
    assert MODEL_SNAPSHOT_DIR.name == MODEL_REVISION, MODEL_SNAPSHOT_DIR
    MODEL_SNAPSHOT_SOURCE = "pinned_hugging_face_revision"

VERIFIED_MODEL_FILES_SHA256 = {}
for relative_name, expected_sha256 in MODEL_FILES_SHA256.items():
    model_file = MODEL_SNAPSHOT_DIR / relative_name
    assert model_file.is_file(), f"Missing pinned model file: {relative_name}"
    actual_sha256 = sha256_file(model_file)
    assert actual_sha256 == expected_sha256, (
        relative_name,
        expected_sha256,
        actual_sha256,
    )
    VERIFIED_MODEL_FILES_SHA256[relative_name] = actual_sha256

MODEL_SNAPSHOT_FINGERPRINT = hashlib.sha256(
    json.dumps(
        VERIFIED_MODEL_FILES_SHA256, sort_keys=True
    ).encode("utf-8")
).hexdigest()
MODEL_LOAD_SOURCE = str(MODEL_SNAPSHOT_DIR)
FROM_PRETRAINED_LOCATION_KWARGS = {"local_files_only": True}
print(
    "Verified pinned model snapshot:",
    MODEL_SNAPSHOT_SOURCE,
    MODEL_SNAPSHOT_FINGERPRINT,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_LOAD_SOURCE,
    src_lang=SOURCE_LANGUAGE_CODE,
    **FROM_PRETRAINED_LOCATION_KWARGS,
)

requested_language_codes = [SOURCE_LANGUAGE_CODE, *TARGETS]
language_token_ids = {
    code: tokenizer.convert_tokens_to_ids(code)
    for code in requested_language_codes
}
for code, token_id in language_token_ids.items():
    assert token_id != tokenizer.unk_token_id, f"Unknown language token: {code}"
    assert tokenizer.convert_ids_to_tokens(token_id) == code
assert len(set(language_token_ids.values())) == len(language_token_ids)
print("Language token IDs:", language_token_ids)


def token_lengths(row):
    source_ids = tokenizer(
        row["source"], add_special_tokens=True, truncation=False
    )["input_ids"]
    tokenizer.tgt_lang = row["target_language"]
    target_ids = tokenizer(
        text_target=row["reference"],
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]
    return len(source_ids), len(target_ids)


def eligible_rows(rows):
    accepted = []
    rejected = []
    for row in rows:
        source_length, target_length = token_lengths(row)
        enriched = {
            **row,
            "_source_tokens": source_length,
            "_target_tokens": target_length,
            "_source_hash": normalized_source_hash(row["source"]),
        }
        if (
            source_length <= MAX_SOURCE_TOKENS
            and target_length <= MAX_TARGET_TOKENS
        ):
            accepted.append(enriched)
        else:
            rejected.append(enriched)
    return accepted, rejected


eligible_ntrex, rejected_ntrex = eligible_rows(ntrex_rows)
eligible_wmt, rejected_wmt = eligible_rows(wmt_rows)
eligible_tico, rejected_tico = eligible_rows(tico_rows)
print("Overlength rejections:", {
    "NTREX": len(rejected_ntrex),
    "WMT24++": len(rejected_wmt),
    "TICO-19": len(rejected_tico),
})


def stable_take(rows, n: int, namespace: str):
    ranked = sorted(
        rows, key=lambda row: rank_key(namespace, row["example_id"])
    )
    assert len(ranked) >= n, f"{namespace}: need {n}, found {len(ranked)}"
    return ranked[:n]


training_cells = {
    f"ntrex_news_{code}": [
        row
        for row in eligible_ntrex
        if row["split"] == "train" and row["target_language"] == code
    ]
    for code in TARGETS
}
training_cells.update(
    {
        f"wmt24pp_{domain}_zho_Hans": [
            row
            for row in eligible_wmt
            if row["split"] == "train" and row["domain"] == domain
        ]
        for domain in ("social", "speech", "literary")
    }
)
B = min(80, min(len(rows) for rows in training_cells.values()))
assert B > 0

zh_news_train = training_cells["ntrex_news_zho_Hans"]
shared_zh_news = stable_take(zh_news_train, B, "arm|shared_zh_news")
shared_hashes = {row["_source_hash"] for row in shared_zh_news}

control_candidates = [
    row for row in zh_news_train if row["_source_hash"] not in shared_hashes
]
control_extra = stable_take(
    control_candidates, 3 * B, "arm|control_extra_zh_news"
)

multilingual_rows = list(shared_zh_news)
multilingual_used_hashes = set(shared_hashes)
for code in ("guj_Gujr", "kat_Geor", "tam_Taml"):
    candidates = [
        row
        for row in training_cells[f"ntrex_news_{code}"]
        if row["_source_hash"] not in multilingual_used_hashes
    ]
    selected = stable_take(candidates, B, f"arm|multilingual|{code}")
    multilingual_rows.extend(selected)
    multilingual_used_hashes.update(row["_source_hash"] for row in selected)

multidomain_rows = list(shared_zh_news)
for domain in ("social", "speech", "literary"):
    multidomain_rows.extend(
        stable_take(
            training_cells[f"wmt24pp_{domain}_zho_Hans"],
            B,
            f"arm|multidomain|{domain}",
        )
    )

proposed_arms = {
    "mandarin_news_control": [*shared_zh_news, *control_extra],
    "multilingual": multilingual_rows,
    "multi_domain": multidomain_rows,
}
assert all(len(rows) == 4 * B for rows in proposed_arms.values())
for rows in proposed_arms.values():
    assert len({row["example_id"] for row in rows}) == 4 * B
shared_ids = {row["example_id"] for row in shared_zh_news}
assert all(
    shared_ids <= {row["example_id"] for row in rows}
    for rows in proposed_arms.values()
)


def proposed_arm_stats(rows):
    examples = len(rows)
    source_tokens = sum(row["_source_tokens"] for row in rows)
    target_tokens = sum(row["_target_tokens"] for row in rows)
    updates_per_epoch = math.ceil(examples / (4 * 4))
    return {
        "examples": examples,
        "source_subword_tokens": source_tokens,
        "target_subword_tokens": target_tokens,
        "non_padding_tokens": source_tokens + target_tokens,
        "padded_tensor_tokens_per_epoch": examples
        * (MAX_SOURCE_TOKENS + MAX_TARGET_TOKENS),
        "per_device_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "epochs": 3,
        "optimizer_steps_per_seed": updates_per_epoch * 3,
        "training_seeds": TRAINING_SEEDS,
        "runtime_seconds": None,
        "peak_gpu_memory_bytes": None,
    }


arm_statistics = {
    name: proposed_arm_stats(rows) for name, rows in proposed_arms.items()
}

# Use the same 25 aligned NTREX segment indices across all targets.
test_indices_by_language = {
    code: {
        row["segment_index"]
        for row in eligible_ntrex
        if row["split"] == "test" and row["target_language"] == code
    }
    for code in TARGETS
}
common_ntrex_test_indices = set.intersection(
    *test_indices_by_language.values()
)
ranked_indices = sorted(
    common_ntrex_test_indices,
    key=lambda value: rank_key("demo|ntrex|aligned", str(value)),
)
selected_ntrex_indices = ranked_indices[
    : SAMPLE_COUNTS["ntrex_per_language"]
]
selected_ntrex = [
    row
    for row in eligible_ntrex
    if row["segment_index"] in selected_ntrex_indices
    and row["target_language"] in TARGETS
]
assert len(selected_ntrex) == len(TARGETS) * len(selected_ntrex_indices)

selected_wmt = []
for domain in ("social", "speech", "literary"):
    candidates = [
        row
        for row in eligible_wmt
        if row["split"] == "test" and row["domain"] == domain
    ]
    selected_wmt.extend(
        stable_take(
            candidates,
            SAMPLE_COUNTS["wmt_per_domain"],
            f"demo|wmt24pp|{domain}",
        )
    )

selected_tico = stable_take(
    eligible_tico,
    SAMPLE_COUNTS["tico"],
    "demo|tico19|official_test",
)

Verified pinned model snapshot: verified_local_override 2798113a43e6f62ab39bd106928a47acecb740b5bf3b914ed44b73366539f071


Language token IDs: {'eng_Latn': 256047, 'guj_Gujr': 256064, 'kat_Geor': 256086, 'tam_Taml': 256170, 'zho_Hans': 256200}


Overlength rejections: {'NTREX': 0, 'WMT24++': 1, 'TICO-19': 1}


In [6]:
COUNTERFACTUAL_PAIRS = [
    {
        "pair_id": "cf01_number",
        "category": "number",
        "original": "The committee will increase the 2026 budget by 12 percent.",
        "changed": "The committee will increase the 2026 budget by 7 percent.",
        "original_reference": "委员会将把2026年的预算提高12%。",
        "changed_reference": "委员会将把2026年的预算提高7%。",
        "old_markers": ["12"],
        "new_markers": ["7"],
    },
    {
        "pair_id": "cf02_year",
        "category": "date",
        "original": "The new clinic will open in September 2026.",
        "changed": "The new clinic will open in September 2027.",
        "original_reference": "新诊所将于2026年9月开业。",
        "changed_reference": "新诊所将于2027年9月开业。",
        "old_markers": ["2026"],
        "new_markers": ["2027"],
    },
    {
        "pair_id": "cf03_city",
        "category": "entity",
        "original": "The research team will meet in Toronto next week.",
        "changed": "The research team will meet in Vancouver next week.",
        "original_reference": "研究团队将于下周在多伦多会面。",
        "changed_reference": "研究团队将于下周在温哥华会面。",
        "old_markers": ["多伦多"],
        "new_markers": ["温哥华"],
    },
    {
        "pair_id": "cf04_calendar_date",
        "category": "date",
        "original": "The hearing is scheduled for March 14.",
        "changed": "The hearing is scheduled for April 18.",
        "original_reference": "听证会定于3月14日举行。",
        "changed_reference": "听证会定于4月18日举行。",
        "old_markers": ["3月14", "3 月 14"],
        "new_markers": ["4月18", "4 月 18"],
    },
    {
        "pair_id": "cf05_decision",
        "category": "polarity",
        "original": "The board approved the proposal after the review.",
        "changed": "The board rejected the proposal after the review.",
        "original_reference": "董事会在审查后批准了该提案。",
        "changed_reference": "董事会在审查后否决了该提案。",
        "old_markers": ["批准", "通过"],
        "new_markers": ["否决", "拒绝", "驳回"],
    },
    {
        "pair_id": "cf06_money",
        "category": "number",
        "original": "The program received 3.5 million dollars in funding.",
        "changed": "The program received 2.1 million dollars in funding.",
        "original_reference": "该项目获得了350万美元的资助。",
        "changed_reference": "该项目获得了210万美元的资助。",
        "old_markers": ["3.5", "350万"],
        "new_markers": ["2.1", "210万"],
    },
    {
        "pair_id": "cf07_organization",
        "category": "entity",
        "original": "The World Health Organization released the guidance.",
        "changed": "UNICEF released the guidance.",
        "original_reference": "世界卫生组织发布了这项指南。",
        "changed_reference": "联合国儿童基金会发布了这项指南。",
        "old_markers": ["世界卫生组织", "世卫组织"],
        "new_markers": ["联合国儿童基金会", "UNICEF"],
    },
    {
        "pair_id": "cf08_direction",
        "category": "polarity",
        "original": "Hospital admissions increased during the week.",
        "changed": "Hospital admissions decreased during the week.",
        "original_reference": "本周住院人数有所增加。",
        "changed_reference": "本周住院人数有所减少。",
        "old_markers": ["增加", "上升"],
        "new_markers": ["减少", "下降"],
    },
    {
        "pair_id": "cf09_weekday",
        "category": "date",
        "original": "The report will be released on Monday.",
        "changed": "The report will be released on Thursday.",
        "original_reference": "该报告将于星期一发布。",
        "changed_reference": "该报告将于星期四发布。",
        "old_markers": ["星期一", "周一"],
        "new_markers": ["星期四", "周四"],
    },
    {
        "pair_id": "cf10_patients",
        "category": "number",
        "original": "The trial enrolled 18 patients from the region.",
        "changed": "The trial enrolled 11 patients from the region.",
        "original_reference": "该试验招募了该地区的18名患者。",
        "changed_reference": "该试验招募了该地区的11名患者。",
        "old_markers": ["18"],
        "new_markers": ["11"],
    },
][: SAMPLE_COUNTS["counterfactual_pairs"]]

counterfactual_rows = []
counterfactual_metadata = {}
for pair in COUNTERFACTUAL_PAIRS:
    for variant in ("original", "changed"):
        example_id = f"counterfactual:{pair['pair_id']}:{variant}"
        row = {
            "example_id": example_id,
            "document_id": pair["pair_id"],
            "source": pair[variant],
            "reference": pair[f"{variant}_reference"],
            "target_language": "zho_Hans",
            "domain": "controlled_counterfactual",
            "dataset": "counterfactual",
            "split": "hand_constructed",
        }
        source_length, target_length = token_lengths(row)
        assert source_length <= MAX_SOURCE_TOKENS
        assert target_length <= MAX_TARGET_TOKENS
        row.update(
            {
                "_source_tokens": source_length,
                "_target_tokens": target_length,
                "_source_hash": normalized_source_hash(row["source"]),
            }
        )
        counterfactual_rows.append(row)
        counterfactual_metadata[example_id] = {
            "pair_id": pair["pair_id"],
            "variant": variant,
            "category": pair["category"],
            "old_markers": pair["old_markers"],
            "new_markers": pair["new_markers"],
        }

demo_rows = [
    *selected_ntrex,
    *selected_wmt,
    *selected_tico,
    *counterfactual_rows,
]
assert len({row["example_id"] for row in demo_rows}) == len(demo_rows)
assert all(row["source"] and row["reference"] for row in demo_rows)

# Enforce global training-to-evaluation separation across every corpus,
# including TICO-19 and the constructed counterfactual examples.
evaluation_source_hashes = {row["_source_hash"] for row in demo_rows}
arm_to_evaluation_hash_overlap = {
    name: sorted(
        {row["_source_hash"] for row in rows}
        & evaluation_source_hashes
    )
    for name, rows in proposed_arms.items()
}
assert all(
    not overlap
    for overlap in arm_to_evaluation_hash_overlap.values()
), arm_to_evaluation_hash_overlap

selected_counts = Counter(
    (row["dataset"], row["target_language"], row["domain"])
    for row in demo_rows
)
print("Frozen B:", B)
print("Eligible training cells:", {
    key: len(value) for key, value in training_cells.items()
})
print("Proposed arm statistics:")
display(pd.DataFrame(arm_statistics).T)
print("Demonstration cells:")
display(
    pd.DataFrame(
        [
            {
                "dataset": key[0],
                "target_language": key[1],
                "domain": key[2],
                "n": value,
            }
            for key, value in sorted(selected_counts.items())
        ]
    )
)

Frozen B: 77
Eligible training cells: {'ntrex_news_guj_Gujr': 1417, 'ntrex_news_kat_Geor': 1417, 'ntrex_news_tam_Taml': 1417, 'ntrex_news_zho_Hans': 1417, 'wmt24pp_social_zho_Hans': 281, 'wmt24pp_speech_zho_Hans': 77, 'wmt24pp_literary_zho_Hans': 117}
Proposed arm statistics:


,examples,source_subword_tokens,target_subword_tokens,non_padding_tokens,padded_tensor_tokens_per_epoch,per_device_batch_size,gradient_accumulation_steps,epochs,optimizer_steps_per_seed,training_seeds,runtime_seconds,peak_gpu_memory_bytes
mandarin_news_control,308,9733,10902,20635,157696,4,4,3,60,"[13, 42, 73]",None,None
multilingual,308,10099,13855,23954,157696,4,4,3,60,"[13, 42, 73]",None,None
multi_domain,308,18024,18886,36910,157696,4,4,3,60,"[13, 42, 73]",None,None


Demonstration cells:


,dataset,target_language,domain,n
0,counterfactual,zho_Hans,controlled_counterfactual,20
1,ntrex128,guj_Gujr,news,25
2,ntrex128,kat_Geor,news,25
3,ntrex128,tam_Taml,news,25
4,ntrex128,zho_Hans,news,25
5,tico19,zho_Hans,medical,25
6,wmt24pp,zho_Hans,literary,15
7,wmt24pp,zho_Hans,social,15
8,wmt24pp,zho_Hans,speech,15


## 5. Load NLLB and assert language/BOS mappings

In [7]:
set_seed(SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    load_dtype = torch.float16
else:
    load_dtype = torch.float32

load_started = time.perf_counter()
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_LOAD_SOURCE,
    torch_dtype=load_dtype,
    low_cpu_mem_usage=True,
    **FROM_PRETRAINED_LOCATION_KWARGS,
)
model.to(DEVICE)
model.eval()
assert VERIFIED_MODEL_FILES_SHA256 == MODEL_FILES_SHA256
assert MODEL_SNAPSHOT_FINGERPRINT
assert max(language_token_ids.values()) < model.config.vocab_size
assert model.config.decoder_start_token_id is not None

for code in TARGETS:
    forced_bos = language_token_ids[code]
    assert tokenizer.convert_ids_to_tokens(forced_bos) == code

MODEL_LOAD_SECONDS = time.perf_counter() - load_started
print(
    f"Loaded {MODEL_ID}@{MODEL_REVISION} on {DEVICE} "
    f"in {MODEL_LOAD_SECONDS:.1f}s ({load_dtype})."
)

Loaded facebook/nllb-200-distilled-600M@f8d333a098d19b4fd9a8b18f94170487ad3f821d on cpu in 2.9s (torch.float32).


## 6. Deterministic zero-shot translation (beam size 4)

In [8]:
GENERATION_CONFIG = {
    "num_beams": NUM_BEAMS,
    "do_sample": False,
    "max_new_tokens": MAX_TARGET_TOKENS,
    "early_stopping": True,
}
GENERATION_CONFIG_HASH = hashlib.sha256(
    json.dumps(GENERATION_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()
DEMO_INPUT_SHA256 = hashlib.sha256(
    json.dumps(
        [
            {
                "example_id": row["example_id"],
                "target_language": row["target_language"],
                "source_sha256": row["_source_hash"],
            }
            for row in demo_rows
        ],
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()
CACHE_SCHEMA_VERSION = 2
INFERENCE_CACHE_PATH = DATA_DIR / "zero_shot_predictions_cache.json"
inference_cache_identity = {
    "schema_version": CACHE_SCHEMA_VERSION,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_snapshot_fingerprint": MODEL_SNAPSHOT_FINGERPRINT,
    "verified_model_files_sha256": VERIFIED_MODEL_FILES_SHA256,
    "generation_config_hash": GENERATION_CONFIG_HASH,
    "demo_input_sha256": DEMO_INPUT_SHA256,
}
inference_cache_key = hashlib.sha256(
    json.dumps(
        inference_cache_identity,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()
expected_prediction_ids = {row["example_id"] for row in demo_rows}


def predictions_sha256(predictions):
    return hashlib.sha256(
        json.dumps(
            predictions, ensure_ascii=False, sort_keys=True
        ).encode("utf-8")
    ).hexdigest()


def translate_group(rows, target_language: str):
    batch_size = 8 if DEVICE.type == "cuda" else 2
    forced_bos_token_id = language_token_ids[target_language]
    assert tokenizer.convert_ids_to_tokens(forced_bos_token_id) == target_language
    outputs = {}
    for start in range(0, len(rows), batch_size):
        batch = rows[start : start + batch_size]
        encoded = tokenizer(
            [row["source"] for row in batch],
            return_tensors="pt",
            padding=True,
            truncation=False,
        )
        assert int(encoded["attention_mask"].sum(dim=1).max()) <= MAX_SOURCE_TOKENS
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                forced_bos_token_id=forced_bos_token_id,
                **GENERATION_CONFIG,
            )
        decoded = tokenizer.batch_decode(
            generated, skip_special_tokens=True
        )
        assert len(decoded) == len(batch)
        for row, prediction in zip(batch, decoded):
            prediction = normalize_text(prediction)
            assert prediction, f"Empty output for {row['example_id']}"
            outputs[row["example_id"]] = prediction
    return outputs


cached_inference = None
if INFERENCE_CACHE_PATH.exists():
    candidate = json.loads(INFERENCE_CACHE_PATH.read_text("utf-8"))
    candidate_predictions = candidate.get("predictions_by_id")
    cache_valid = (
        candidate.get("schema_version") == CACHE_SCHEMA_VERSION
        and candidate.get("cache_key") == inference_cache_key
        and candidate.get("identity") == inference_cache_identity
        and isinstance(candidate_predictions, dict)
        and set(candidate_predictions) == expected_prediction_ids
        and all(
            isinstance(prediction, str)
            and prediction
            and normalize_text(prediction) == prediction
            for prediction in candidate_predictions.values()
        )
        and candidate.get("predictions_sha256")
        == predictions_sha256(candidate_predictions)
    )
    if cache_valid:
        cached_inference = candidate

if cached_inference is not None:
    predictions_by_id = cached_inference["predictions_by_id"]
    INFERENCE_SECONDS = float(cached_inference["inference_seconds"])
    PEAK_GPU_MEMORY_BYTES = cached_inference["peak_gpu_memory_bytes"]
    INFERENCE_CACHE_USED = True
    print("Loaded validated temporary inference cache.")
else:
    inference_started = time.perf_counter()
    predictions_by_id = {}
    for target_language in TARGETS:
        target_rows = [
            row
            for row in demo_rows
            if row["target_language"] == target_language
        ]
        predictions_by_id.update(
            translate_group(target_rows, target_language)
        )
        print(
            f"Translated {len(target_rows)} examples to "
            f"{target_language}."
        )
    INFERENCE_SECONDS = time.perf_counter() - inference_started
    PEAK_GPU_MEMORY_BYTES = (
        int(torch.cuda.max_memory_allocated())
        if DEVICE.type == "cuda"
        else None
    )
    INFERENCE_CACHE_USED = False
    INFERENCE_CACHE_PATH.write_text(
        json.dumps(
            {
                "schema_version": CACHE_SCHEMA_VERSION,
                "cache_key": inference_cache_key,
                "identity": inference_cache_identity,
                "inference_seconds": INFERENCE_SECONDS,
                "peak_gpu_memory_bytes": PEAK_GPU_MEMORY_BYTES,
                "predictions_by_id": predictions_by_id,
                "predictions_sha256": predictions_sha256(predictions_by_id),
            },
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

assert len(predictions_by_id) == len(demo_rows)
assert all(predictions_by_id[row["example_id"]] for row in demo_rows)
print(f"Inference completed in {INFERENCE_SECONDS:.1f}s.")

Translated 25 examples to guj_Gujr.


Translated 25 examples to kat_Geor.


Translated 25 examples to tam_Taml.


Translated 115 examples to zho_Hans.
Inference completed in 434.2s.


## 7. Metrics and source-faithfulness diagnostics

In [9]:
# The digit-based diagnostic normalizes Unicode, separators, and English
# month names. It intentionally does not pretend to resolve every
# language-specific written-out number.
MONTHS = {
    "january": "1",
    "february": "2",
    "march": "3",
    "april": "4",
    "may": "5",
    "june": "6",
    "july": "7",
    "august": "8",
    "september": "9",
    "october": "10",
    "november": "11",
    "december": "12",
}
NUMBER_RE = re.compile(r"(?<!\d)[+-]?\d+(?:[.,]\d+)*(?!\d)")


def extract_numeric_facts(text: str):
    normalized = unicodedata.normalize("NFKC", text).casefold()
    for month, number in MONTHS.items():
        normalized = re.sub(rf"\b{month}\b", number, normalized)
    facts = []
    for match in NUMBER_RE.finditer(normalized):
        token = match.group(0).replace(",", "")
        if token.startswith("+"):
            token = token[1:]
        try:
            value = format(float(token), ".12g") if "." in token else str(int(token))
        except ValueError:
            continue
        facts.append(value)
    return facts


def multiset_overlap(left, right):
    left_counter = Counter(left)
    right_counter = Counter(right)
    return sum((left_counter & right_counter).values())


def numeric_diagnostic(source: str, prediction: str):
    source_facts = extract_numeric_facts(source)
    prediction_facts = extract_numeric_facts(prediction)
    matched = multiset_overlap(source_facts, prediction_facts)
    spurious = max(0, len(prediction_facts) - matched)
    return {
        "source_fact_count": len(source_facts),
        "prediction_fact_count": len(prediction_facts),
        "matched_fact_count": matched,
        "spurious_fact_count": spurious,
        "number_recall": (
            matched / len(source_facts) if source_facts else None
        ),
        "spurious_number_rate": (
            spurious / len(prediction_facts) if prediction_facts else 0.0
        ),
    }


# Fail-fast metric fixtures.
fixture_hypotheses = ["The clinic opens on 14 March 2027."]
fixture_references = ["The clinic opens on 14 March 2027."]
fixture_chrf = CHRF(word_order=2).corpus_score(
    fixture_hypotheses, [fixture_references]
).score
fixture_bleu = BLEU(
    tokenize="intl", effective_order=True
).corpus_score(fixture_hypotheses, [fixture_references]).score
assert abs(fixture_chrf - 100.0) < 1e-9
assert abs(fixture_bleu - 100.0) < 1e-9
fixture_numeric = numeric_diagnostic(
    "The clinic opens on 14 March 2027.",
    "诊所将于2027年3月14日开业。",
)
assert fixture_numeric["matched_fact_count"] == 3
assert fixture_numeric["spurious_fact_count"] == 0

row_by_id = {row["example_id"]: row for row in demo_rows}
prediction_rows = [
    {
        "example_id": row["example_id"],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "prediction": predictions_by_id[row["example_id"]],
        "generation_config_hash": GENERATION_CONFIG_HASH,
    }
    for row in demo_rows
]

scored_rows = [
    row for row in demo_rows if row["dataset"] != "counterfactual"
]
metric_groups = defaultdict(list)
for row in scored_rows:
    metric_groups[
        (row["dataset"], row["target_language"], row["domain"])
    ].append(row)

metric_rows = []
for (dataset, target_language, domain), rows in sorted(metric_groups.items()):
    hypotheses = [predictions_by_id[row["example_id"]] for row in rows]
    references = [row["reference"] for row in rows]
    chrf_metric = CHRF(word_order=2)
    chrf_score = chrf_metric.corpus_score(hypotheses, [references]).score
    bleu_tokenizer = TARGETS[target_language]["bleu_tokenizer"]
    bleu_metric = BLEU(tokenize=bleu_tokenizer, effective_order=True)
    bleu_score = bleu_metric.corpus_score(hypotheses, [references]).score

    diagnostics = [
        numeric_diagnostic(
            row["source"], predictions_by_id[row["example_id"]]
        )
        for row in rows
    ]
    source_fact_count = sum(item["source_fact_count"] for item in diagnostics)
    prediction_fact_count = sum(
        item["prediction_fact_count"] for item in diagnostics
    )
    matched_fact_count = sum(item["matched_fact_count"] for item in diagnostics)
    spurious_fact_count = sum(item["spurious_fact_count"] for item in diagnostics)
    metric_rows.append(
        {
            "model_id": MODEL_ID,
            "dataset": dataset,
            "target_language": target_language,
            "domain": domain,
            "n": len(rows),
            "chrf2": round(chrf_score, 4),
            "bleu": round(bleu_score, 4),
            "comet": None,
            "number_recall": (
                round(matched_fact_count / source_fact_count, 4)
                if source_fact_count
                else None
            ),
            "spurious_number_rate": (
                round(spurious_fact_count / prediction_fact_count, 4)
                if prediction_fact_count
                else 0.0
            ),
            "source_number_fact_count": source_fact_count,
            "prediction_number_fact_count": prediction_fact_count,
            "matched_number_fact_count": matched_fact_count,
            "spurious_number_fact_count": spurious_fact_count,
            "bleu_tokenizer": bleu_tokenizer,
            "bleu_signature": str(bleu_metric.get_signature()),
            "chrf_signature": str(chrf_metric.get_signature()),
        }
    )

metrics_df = pd.DataFrame(metric_rows).sort_values(
    ["dataset", "target_language", "domain"]
)
display(
    metrics_df[
        [
            "dataset",
            "target_language",
            "domain",
            "n",
            "chrf2",
            "bleu",
            "number_recall",
            "spurious_number_rate",
        ]
    ]
)

,dataset,target_language,domain,n,chrf2,bleu,number_recall,spurious_number_rate
0,ntrex128,guj_Gujr,news,25,48.0125,18.0972,0.8235,0.1250
1,ntrex128,kat_Geor,news,25,47.0692,18.2385,0.7647,0.1875
2,ntrex128,tam_Taml,news,25,45.9723,11.6439,0.8235,0.0667
3,ntrex128,zho_Hans,news,25,22.7690,29.6883,0.7059,0.1429
4,tico19,zho_Hans,medical,25,34.6154,40.5100,0.9024,0.0750
5,wmt24pp,zho_Hans,literary,15,12.1349,13.1454,0.1250,0.6667
6,wmt24pp,zho_Hans,social,15,28.1602,22.8247,0.3333,0.0000
7,wmt24pp,zho_Hans,speech,15,17.4603,23.4543,0.8000,0.1429


In [10]:
def compact_for_marker_matching(text: str) -> str:
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", text))


def marker_pattern(marker: str) -> str:
    compact_marker = compact_for_marker_matching(marker)
    escaped = re.escape(compact_marker)
    if re.fullmatch(r"[0-9]+(?:\.[0-9]+)?", compact_marker):
        return rf"(?<![0-9.]){escaped}(?![0-9.])"
    return escaped


def contains_any(text: str, markers) -> bool:
    compact = compact_for_marker_matching(text)
    return any(
        re.search(marker_pattern(marker), compact) is not None
        for marker in markers
    )


def mask_markers(text: str, markers) -> str:
    masked = compact_for_marker_matching(text)
    for marker in sorted(markers, key=len, reverse=True):
        masked = re.sub(marker_pattern(marker), "<FACT>", masked)
    return masked


counterfactual_results = []
for pair in COUNTERFACTUAL_PAIRS:
    original_id = f"counterfactual:{pair['pair_id']}:original"
    changed_id = f"counterfactual:{pair['pair_id']}:changed"
    original_prediction = predictions_by_id[original_id]
    changed_prediction = predictions_by_id[changed_id]
    old_fact_in_original = contains_any(
        original_prediction, pair["old_markers"]
    )
    new_fact_absent_original = not contains_any(
        original_prediction, pair["new_markers"]
    )
    new_fact_appears = contains_any(
        changed_prediction, pair["new_markers"]
    )
    old_fact_disappears = not contains_any(
        changed_prediction, pair["old_markers"]
    )
    markers = [*pair["old_markers"], *pair["new_markers"]]
    masked_original = mask_markers(original_prediction, markers)
    masked_changed = mask_markers(changed_prediction, markers)
    unrelated_chrf2 = CHRF(word_order=2).sentence_score(
        masked_changed, [masked_original]
    ).score
    unrelated_stable = unrelated_chrf2 >= 70.0
    counterfactual_results.append(
        {
            "pair_id": pair["pair_id"],
            "category": pair["category"],
            "old_fact_in_original": old_fact_in_original,
            "new_fact_absent_original": new_fact_absent_original,
            "new_fact_appears": new_fact_appears,
            "old_fact_disappears": old_fact_disappears,
            "unrelated_chrf2": round(unrelated_chrf2, 2),
            "unrelated_stable_at_70": unrelated_stable,
            "all_three_criteria": (
                old_fact_in_original
                and new_fact_absent_original
                and new_fact_appears
                and old_fact_disappears
                and unrelated_stable
            ),
        }
    )

counterfactual_df = pd.DataFrame(counterfactual_results)
display(counterfactual_df)
print(
    "All-three sensitivity:",
    f"{int(counterfactual_df['all_three_criteria'].sum())}/"
    f"{len(counterfactual_df)}",
)

,pair_id,category,old_fact_in_original,new_fact_absent_original,new_fact_appears,old_fact_disappears,unrelated_chrf2,unrelated_stable_at_70,all_three_criteria
0,cf01_number,number,False,True,True,True,43.84,False,False
1,cf02_year,date,True,True,True,True,100.00,True,True
2,cf03_city,entity,True,True,True,True,61.52,False,False
3,cf04_calendar_date,date,True,True,True,True,100.00,True,True
4,cf05_decision,polarity,True,True,True,True,100.00,True,True
5,cf06_money,number,True,True,False,True,39.09,False,False
6,cf07_organization,entity,True,True,True,True,55.26,False,False
7,cf08_direction,polarity,True,True,True,True,100.00,True,True
8,cf09_weekday,date,True,True,True,True,100.00,True,True
9,cf10_patients,number,True,True,True,True,100.00,True,True


All-three sensitivity: 6/10


## 8. Report-ready tables and Mandarin audit worksheet

In [11]:
# Labels are intentionally explicit and reproducible. Source/reference
# text was inspected during annotation but is not displayed in this
# public notebook output.
MANUAL_AUDIT_LABELS = {
    "ntrex:1655:zho_Hans": {
        "unsupported_addition": False,
        "omission": True,
        "contradiction": True,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "major",
        "notes": "Backbencher is mistranslated, and the output stops before the main claim that the UK would already have left.",
    },
    "ntrex:0044:zho_Hans": {
        "unsupported_addition": False,
        "omission": True,
        "contradiction": True,
        "entity_or_number_mutation": True,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "major",
        "notes": "Shark and lobster season are weakened to fish/fish season; the diving context and Saturday are omitted.",
    },
    "ntrex:0175:zho_Hans": {
        "unsupported_addition": False,
        "omission": True,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "minor",
        "notes": "The entire Cromwell clause is omitted.",
    },
    "ntrex:1320:zho_Hans": {
        "unsupported_addition": False,
        "omission": True,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "major",
        "notes": "The expected autumn decisions are omitted from the truncated output.",
    },
    "ntrex:0816:zho_Hans": {
        "unsupported_addition": False,
        "omission": True,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "minor",
        "notes": "The example mineral uranium is missing and the sentence ends after an incomplete 'such as'.",
    },
    "ntrex:0960:zho_Hans": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "none",
        "notes": "Meaning and attribution are preserved.",
    },
    "ntrex:1683:zho_Hans": {
        "unsupported_addition": False,
        "omission": True,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "minor",
        "notes": "The Boston Globe attribution is omitted; the reported event is retained.",
    },
    "ntrex:1292:zho_Hans": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "none",
        "notes": "The plan, credibility claim, and EU relationship are preserved.",
    },
    "ntrex:1667:zho_Hans": {
        "unsupported_addition": False,
        "omission": True,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "major",
        "notes": "The public-opinion clause and activists' electoral role are omitted.",
    },
    "ntrex:1970:zho_Hans": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "none",
        "notes": "Selection, gratitude, and belief are preserved.",
    },
    "tico19:PubMed_11:1058": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "none",
        "notes": "Disease and virus names, dates, and abbreviations are preserved.",
    },
    "tico19:PubMed_8:599": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": True,
        "entity_or_number_mutation": True,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "major",
        "notes": "Masked palm civet and raccoon dog are mistranslated as a palm tree and a generic dog.",
    },
    "tico19:Wikipedia_handpicked_4:1712": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": True,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "major",
        "notes": "Administering medicine is changed to taking medicine, altering the action and likely actor.",
    },
    "tico19:PubMed_8:564": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": True,
        "severity": "minor",
        "notes": "Core comparison and lower ratio are present, but the Chinese phrasing is awkward.",
    },
    "tico19:PubMed_7:213": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "none",
        "notes": "Rapid change and unresolved scope/severity are preserved.",
    },
    "tico19:Wikipedia_handpicked_3:1672": {
        "unsupported_addition": False,
        "omission": True,
        "contradiction": False,
        "entity_or_number_mutation": True,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "major",
        "notes": "The output truncates Shenzhen, omits the Huo-Yan lab relation, and drops the total of 12 cities.",
    },
    "tico19:PubMed_7:412": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": True,
        "entity_or_number_mutation": True,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "major",
        "notes": "Mid-turbinate and throat swabs are mistranslated as a wheel and throat scan, corrupting clinical sampling details.",
    },
    "tico19:Wikipedia_handpicked_7:2033": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "none",
        "notes": "The listed occupations and exposure condition are preserved.",
    },
    "tico19:PubMed_8:542": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": True,
        "entity_or_number_mutation": True,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "major",
        "notes": "Renal failure is changed to an incomplete generic organ-failure phrase; dates and age remain correct.",
    },
    "tico19:Wikipedia_handpicked_1:1524": {
        "unsupported_addition": False,
        "omission": False,
        "contradiction": False,
        "entity_or_number_mutation": False,
        "non_translation": False,
        "repetition": False,
        "fluency_only": False,
        "severity": "none",
        "notes": "Russia and the geopolitical/diplomatic charm-offensive claim are preserved.",
    },
}

mandarin_news = [
    row
    for row in selected_ntrex
    if row["target_language"] == "zho_Hans"
]
audit_news = stable_take(mandarin_news, min(10, len(mandarin_news)), "audit|news")
audit_medical = stable_take(
    selected_tico, min(10, len(selected_tico)), "audit|medical"
)
audit_rows_private = [*audit_news, *audit_medical]

audit_rows = []
for row in audit_rows_private:
    example_id = row["example_id"]
    labels = MANUAL_AUDIT_LABELS.get(example_id, {})
    automatic = numeric_diagnostic(
        row["source"], predictions_by_id[example_id]
    )
    audit_rows.append(
        {
            "example_id": example_id,
            "dataset": row["dataset"],
            "domain": row["domain"],
            "prediction": predictions_by_id[example_id],
            "auto_number_recall": automatic["number_recall"],
            "auto_spurious_number_rate": automatic["spurious_number_rate"],
            "unsupported_addition": labels.get("unsupported_addition", ""),
            "omission": labels.get("omission", ""),
            "contradiction": labels.get("contradiction", ""),
            "entity_or_number_mutation": labels.get(
                "entity_or_number_mutation", ""
            ),
            "non_translation": labels.get("non_translation", ""),
            "repetition": labels.get("repetition", ""),
            "fluency_only": labels.get("fluency_only", ""),
            "severity": labels.get("severity", ""),
            "review_status": (
                "reviewed" if example_id in MANUAL_AUDIT_LABELS else "pending"
            ),
            "notes": labels.get("notes", ""),
        }
    )

audit_public_preview_df = pd.DataFrame(audit_rows)[
    [
        "example_id",
        "dataset",
        "domain",
        "prediction",
        "severity",
        "review_status",
    ]
]
print("Public audit preview (source/reference text intentionally omitted):")
display(audit_public_preview_df)

Public audit preview (source/reference text intentionally omitted):


,example_id,dataset,domain,prediction,severity,review_status
0,ntrex:1655:zho_Hans,ntrex128,news,"保守派后卫彼得· (Peter Bone) 在伯明翰的游行中表示,如果法拉奇先生是英国脱欧部长,",major,reviewed
1,ntrex:0044:zho_Hans,ntrex128,news,"官员说,在加利福尼亚州鱼季节开赛日,一条鱼袭击并伤害了一名13岁的男孩,",major,reviewed
2,ntrex:0175:zho_Hans,ntrex128,news,"肯定的是,17世纪中叶的冲突塑造了我们国家的后续发展,",minor,reviewed
3,ntrex:1320:zho_Hans,ntrex128,news,"在她的信中,弗里曼女士写道:""在夏季期间,英国和欧盟之间的退出谈判继续进行,",major,reviewed
4,ntrex:0816:zho_Hans,ntrex128,news,"罗杰斯说:""这是一个盖格计数器,用于找到放射性矿物质,如.",minor,reviewed
5,ntrex:0960:zho_Hans,ntrex128,news,"""我的意思是,当然"",肯尼迪说.",none,reviewed
6,ntrex:1683:zho_Hans,ntrex128,news,马萨诸塞州民主党人周六在马萨诸塞州西部的一个市政厅谈到了她的未来.,minor,reviewed
7,ntrex:1292:zho_Hans,ntrex128,news,"一位政府发言人说:""我们已经提出了与欧盟的未来关系的准确可靠计划.",none,reviewed
8,ntrex:1667:zho_Hans,ntrex128,news,"反对派不会投票支持它, 我们的党和活动家不喜欢它,",major,reviewed
9,ntrex:1970:zho_Hans,ntrex128,news,"我非常感谢托马斯选择了我,相信我.",none,reviewed


## 9. Save reproducibility metadata and public artifacts

In [12]:
execution_timestamp = datetime.now(timezone.utc).isoformat()
split_counts = Counter(
    (
        row["dataset"],
        row["target_language"],
        row["domain"],
        row["split"],
    )
    for row in [*eligible_ntrex, *eligible_wmt, *eligible_tico]
)

manifest = {
    "schema_version": "1.0",
    "execution_status": "complete",
    "executed_at_utc": execution_timestamp,
    "deadline_timezone": "America/Toronto",
    "seed": SEED,
    "reduced_fallback": REDUCED_DEMO,
    "sample_counts_requested": SAMPLE_COUNTS,
    "sample_count_total": len(demo_rows),
    "sample_ids": [row["example_id"] for row in demo_rows],
    "model": {
        "id": MODEL_ID,
        "revision": MODEL_REVISION,
        "weights_sha256": MODEL_WEIGHTS_SHA256,
        "verified_files_sha256": VERIFIED_MODEL_FILES_SHA256,
        "snapshot_fingerprint": MODEL_SNAPSHOT_FINGERPRINT,
        "snapshot_source": MODEL_SNAPSHOT_SOURCE,
        "provenance_verified": True,
        "license": "cc-by-nc-4.0",
        "source_language_code": SOURCE_LANGUAGE_CODE,
        "target_language_codes": list(TARGETS),
        "language_token_ids": language_token_ids,
    },
    "generation": {
        **GENERATION_CONFIG,
        "generation_config_hash": GENERATION_CONFIG_HASH,
    },
    "data": {
        "ntrex_revision": NTREX_REVISION,
        "wmt24pp_revision": WMT24PP_REVISION,
        "tico19_archive_sha256": DATA_FILES["tico19_archive"]["sha256"],
        "downloaded_file_sha256": {
            key: sha256_file(path) for key, path in LOCAL_DATA.items()
        },
        "licenses": {
            "ntrex128": "CC BY-SA 4.0",
            "wmt24pp": "Apache-2.0",
            "tico19_translations": "CC0; source licenses vary by row",
        },
        "canary_rows_after_filter": 0,
        "bad_source_rows_after_filter": 0,
        "arm_to_evaluation_source_hash_overlap": {
            name: len(overlap)
            for name, overlap in arm_to_evaluation_hash_overlap.items()
        },
        "deduplication": {
            "ntrex_retained": len(retained_ntrex_indices),
            "wmt24pp_retained": len(dedup_wmt),
        },
        "overlength_rejections": {
            "ntrex": len(rejected_ntrex),
            "wmt24pp": len(rejected_wmt),
            "tico19": len(rejected_tico),
        },
        "eligible_split_counts": [
            {
                "dataset": key[0],
                "target_language": key[1],
                "domain": key[2],
                "split": key[3],
                "n": value,
            }
            for key, value in sorted(split_counts.items())
        ],
    },
    "proposed_lora_experiment": {
        "executed": False,
        "B": B,
        "eligible_training_cell_counts": {
            key: len(value) for key, value in training_cells.items()
        },
        "shared_mandarin_news_example_ids": sorted(shared_ids),
        "arms": {
            name: {
                "example_ids": [row["example_id"] for row in rows],
                "statistics": arm_statistics[name],
            }
            for name, rows in proposed_arms.items()
        },
        "configuration": {
            "lora_r": 8,
            "lora_alpha": 16,
            "lora_dropout": 0.05,
            "target_modules": ["q_proj", "v_proj"],
            "learning_rate": 2e-4,
            "per_device_batch_size": 4,
            "gradient_accumulation_steps": 4,
            "epochs": 3,
            "warmup_ratio": 0.05,
            "max_source_length": MAX_SOURCE_TOKENS,
            "max_target_length": MAX_TARGET_TOKENS,
            "pad_to_declared_maximum": True,
            "training_seeds": TRAINING_SEEDS,
            "checkpoint_rule": "deterministic final checkpoint",
        },
    },
    "metrics": {
        "chrf2_primary": True,
        "bleu_averaged_across_scripts": False,
        "comet_status": "not_run_deadline_safeguard",
        "counterfactual_unrelated_stability_threshold_chrf2": 70.0,
    },
    "counterfactual_results": counterfactual_results,
    "manual_audit": {
        "requested_rows": len(audit_rows),
        "reviewed_rows": sum(
            row["review_status"] == "reviewed" for row in audit_rows
        ),
        "status": (
            "complete"
            if audit_rows
            and all(row["review_status"] == "reviewed" for row in audit_rows)
            else "pending_candidate_review"
        ),
    },
    "runtime": {
        "versions": RUNTIME_VERSIONS,
        "device": str(DEVICE),
        "gpu_name": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
        "model_load_seconds": round(MODEL_LOAD_SECONDS, 3),
        "inference_seconds": round(INFERENCE_SECONDS, 3),
        "peak_gpu_memory_bytes": PEAK_GPU_MEMORY_BYTES,
        "local_snapshot_override_used": bool(local_model_override),
        "validated_inference_cache_used": INFERENCE_CACHE_USED,
        "inference_cache_key_sha256": inference_cache_key,
        "demo_input_sha256": DEMO_INPUT_SHA256,
    },
}

predictions_path = RESULTS_DIR / "demo_predictions.csv"
metrics_path = RESULTS_DIR / "demo_metrics.csv"
audit_path = RESULTS_DIR / "mandarin_audit.csv"
examples_path = RESULTS_DIR / "examples.md"
manifest_path = RESULTS_DIR / "demo_manifest.json"

pd.DataFrame(prediction_rows).to_csv(predictions_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
pd.DataFrame(audit_rows).to_csv(audit_path, index=False)

example_lines = [
    "# Mandarin demonstration examples",
    "",
    "These are a small, rule-selected excerpt. The repository does not "
    "publish bulk source text or references.",
    "The corpus excerpts below are from NTREX-128 (CC BY-SA 4.0) at "
    f"revision `{NTREX_REVISION}`.",
    "",
    "## Corpus examples",
    "",
    "| ID | Domain | English source | Reference | NLLB prediction |",
    "|---|---|---|---|---|",
]
for row in audit_rows_private[:4]:
    safe = lambda value: str(value).replace("|", "\\|").replace("\n", " ")
    example_lines.append(
        "| {id} | {domain} | {source} | {reference} | {prediction} |".format(
            id=safe(row["example_id"]),
            domain=safe(row["domain"]),
            source=safe(row["source"]),
            reference=safe(row["reference"]),
            prediction=safe(predictions_by_id[row["example_id"]]),
        )
    )
example_lines.extend(
    [
        "",
        "## Counterfactual sensitivity",
        "",
        "| Pair | Category | Old fact in original | New fact absent in original | "
        "New fact appears | Old fact disappears | Unrelated chrF++ | All criteria |",
        "|---|---|---:|---:|---:|---:|---:|---:|",
    ]
)
for result in counterfactual_results:
    example_lines.append(
        f"| {result['pair_id']} | {result['category']} | "
        f"{result['old_fact_in_original']} | "
        f"{result['new_fact_absent_original']} | "
        f"{result['new_fact_appears']} | {result['old_fact_disappears']} | "
        f"{result['unrelated_chrf2']:.2f} | "
        f"{result['all_three_criteria']} |"
    )
examples_path.write_text("\n".join(example_lines) + "\n", encoding="utf-8")

manifest["artifact_sha256"] = {
    path.name: sha256_file(path)
    for path in (predictions_path, metrics_path, audit_path, examples_path)
}
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

expected_files = {
    "demo_manifest.json",
    "demo_predictions.csv",
    "demo_metrics.csv",
    "mandarin_audit.csv",
    "examples.md",
}
assert expected_files <= {path.name for path in RESULTS_DIR.iterdir()}
assert len(pd.read_csv(predictions_path)) == len(demo_rows)
assert len(pd.read_csv(metrics_path)) == len(metric_rows)
assert json.loads(manifest_path.read_text("utf-8"))["sample_ids"] == [
    row["example_id"] for row in demo_rows
]

print("Saved:")
for name in sorted(expected_files):
    path = RESULTS_DIR / name
    print(
        f"  {path.relative_to(PROJECT_ROOT)} "
        f"({path.stat().st_size:,} bytes)"
    )

if IN_COLAB:
    archive_path = shutil.make_archive(
        "/content/halo_option1_results", "zip", RESULTS_DIR
    )
    print("Created:", archive_path)

Saved:
  results/demo_manifest.json (54,147 bytes)
  results/demo_metrics.csv (2,030 bytes)
  results/demo_predictions.csv (69,395 bytes)
  results/examples.md (3,053 bytes)
  results/mandarin_audit.csv (5,778 bytes)


## Optional secondary metric: COMET (disabled by default)

In [13]:
RUN_COMET = False
if RUN_COMET:
    raise NotImplementedError(
        "COMET is intentionally outside the deadline-critical path. "
        "Enable only after pinning unbabel-comet and rerunning every "
        "cell; never paste COMET values manually into the artifacts."
    )
else:
    print("COMET skipped by the declared deadline safeguard.")

COMET skipped by the declared deadline safeguard.


## Completion boundary

If every preceding assertion passed, the notebook has executed the
zero-shot demonstration and written its public artifacts. The proposed
LoRA arm manifests are reproducible, but no adapter was trained and no
adapted-system claim is made.